# PAD-ONAP — S3 AI Pipeline: Full Training on CICDDoS2019

**Thesis:** Proactive AI-Driven DDoS Defense (PAD-ONAP)  
**Module:** M2 — AI Detection & Forecast  
**Tracks:**
- **Track A:** XGBoost 7-class classifier (17 entropy features, FGSM adversarial)
- **Track B:** Transformer+LSTM 4-horizon forecaster (+6w/+12w/+18w/+24w)

**Dataset:** CICDDoS2019 (Canadian Institute for Cybersecurity)  
**GPU:** RTX T4 / P100 on Kaggle

---

## Pipeline Flow
```
CICDDoS2019 CSVs
     │
     ▼  [Section 1] Dataset Audit
Row counts, label distribution, class mapping
     │
     ▼  [Section 2] Feature Extraction (streaming, full dataset)
17 entropy/rate features per 5s window → X_train, X_test, y_train, y_test
     │
     ├──▶ [Section 3] Track A: XGBoost 7-class + FGSM → xgboost_v3.json
     │
     └──▶ [Section 4] Track B: Transformer+LSTM 4-horizon → transformer_v3.pt
     │
     ▼  [Section 5] Evaluation Summary + SHAP
```

In [ ]:
# ── Cell 0: Install / upgrade packages ──────────────────────────────────────
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

pip('xgboost>=2.0', 'shap>=0.44', 'scikit-learn>=1.4', 'optuna>=3.6')
print('Packages ready.')

Packages ready.


In [2]:
# ── Cell 1: Imports & GPU check ──────────────────────────────────────────────
import os, glob, warnings, time, json, pickle
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

import xgboost as xgb
import shap
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix,
    balanced_accuracy_score, average_precision_score,
    )
warnings.filterwarnings('ignore')

# ── Paths (adjust DATASET_DIR to your Kaggle dataset mount point) ────────────
DATASET_DIR  = '/kaggle/input/datasets/rodrigorosasilva/cic-ddos2019-30gb-full-dataset-csv-files'
OUTPUT_DIR   = Path('/kaggle/working/pad_onap_v3')
MODELS_DIR   = OUTPUT_DIR / 'models'
DATA_DIR     = OUTPUT_DIR / 'processed'
FIGURES_DIR  = OUTPUT_DIR / 'figures'

for d in [MODELS_DIR, DATA_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── GPU ───────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

XGB_DEVICE = 'cuda' if DEVICE.type == 'cuda' else 'cpu'
try:
    _m = xgb.XGBClassifier(device='cuda', n_estimators=3, verbosity=0)
    _m.fit(np.random.randn(50,5), np.random.randint(0,3,50))
    print('XGBoost GPU: OK')
except Exception:
    XGB_DEVICE = 'cpu'
    print('XGBoost GPU: not available, using CPU')

PyTorch device: cuda
GPU: Tesla T4
VRAM: 15.6 GB
XGBoost GPU: OK


In [ ]:
# ── Cell 1.5: Run mode switch (tune vs baseline) ─────────────────────────────
RUN_MODE = 'baseline'  # 'tune' | 'baseline'

if RUN_MODE not in {'tune', 'baseline'}:
    raise ValueError("RUN_MODE must be either 'tune' or 'baseline'")

if RUN_MODE == 'tune':
    # Run full search once and save best configs.
    RUN_XGB_TUNING = True
    LOAD_XGB_TUNED = False
    SAVE_XGB_TUNED = True

    RUN_LIGHT_TF_SEARCH = True
    RUN_TF_SEARCH = True
    LOAD_TF_TUNED = False
    SAVE_TF_TUNED = True
else:
    # Fast baseline: load best tuned configs and skip search.
    RUN_XGB_TUNING = False
    LOAD_XGB_TUNED = True
    SAVE_XGB_TUNED = False

    RUN_LIGHT_TF_SEARCH = False
    RUN_TF_SEARCH = False
    LOAD_TF_TUNED = True
    SAVE_TF_TUNED = False

print(f'RUN_MODE={RUN_MODE}')
print(f'  XGB -> tune:{RUN_XGB_TUNING}, load:{LOAD_XGB_TUNED}, save:{SAVE_XGB_TUNED}')
print(f'  TF  -> light_search:{RUN_LIGHT_TF_SEARCH}, search:{RUN_TF_SEARCH}, load:{LOAD_TF_TUNED}, save:{SAVE_TF_TUNED}')

---
## Section 1 — Dataset Audit

In [3]:
# ── Cell 2: Dataset Audit (full scan) ────────────────────────────────────────
LABEL_MAP = {
    'BENIGN': 0,
    'DrDoS_UDP': 1, 'UDP': 1, 'UDP-lag': 1, 'UDP-Lag': 1, 'UDPLag': 1,
    'Syn': 2,
    'TFTP': 5,         # TFTP is UDP amplification (not HTTP Flood) → class 5
    'DrDoS_DNS': 5, 'DrDoS_NTP': 5, 'DrDoS_SNMP': 5, 'DrDoS_SSDP': 5,
    'DrDoS_LDAP': 5, 'DrDoS_MSSQL': 5, 'DrDoS_NetBIOS': 5,
    'Portmap': 5, 'LDAP': 5, 'MSSQL': 5, 'NetBIOS': 5,
    'DrDoS_HTTP': 3,   # FIX: HTTP reflection/flood belongs to HTTP_Flood class
}

CLASS_NAMES = {
    0: 'Normal', 1: 'UDP_Flood', 2: 'SYN_Flood',
    3: 'HTTP_Flood', 4: 'ICMP_Flood', 5: 'Amplification', 6: 'Slow_rate',
    # NOTE: class 4 (ICMP_Flood) has no entries in CICDDoS2019 — column stays 0
}

FEATURE_NAMES = [
    'pkt_rate', 'byte_rate', 'src_ip_entropy', 'dst_ip_entropy',
    'src_port_entropy', 'dst_port_entropy', 'proto_dist_tcp',
    'proto_dist_udp', 'proto_dist_icmp', 'syn_ratio', 'fin_ratio',
    'avg_pkt_size', 'pkt_size_std', 'new_flows_rate',
    'flow_duration_mean', 'inter_arrival_mean', 'inter_arrival_std',
]

csv_files = sorted(glob.glob(os.path.join(DATASET_DIR, '**', '*.csv'), recursive=True))
print(f'Found {len(csv_files)} CSV files')
print(f'Total size: {sum(os.path.getsize(f) for f in csv_files)/1e9:.2f} GB\n')

all_labels   = defaultdict(int)
file_summary = []
CHUNKSIZE    = 500_000

for f in csv_files:
    fname     = os.path.basename(f)
    file_rows = 0
    file_lbls = defaultdict(int)
    try:
        for chunk in pd.read_csv(f, chunksize=CHUNKSIZE, low_memory=False, on_bad_lines='skip'):
            chunk.columns = [c.strip() for c in chunk.columns]
            lcol = next((c for c in chunk.columns if c.lower() == 'label'), None)
            if lcol is None: continue
            for lbl, cnt in chunk[lcol].str.strip().value_counts().items():
                file_lbls[lbl] += int(cnt)
                all_labels[lbl] += int(cnt)
            file_rows += len(chunk)
        benign_cnt = file_lbls.get('BENIGN', 0)
        attack_cnt = file_rows - benign_cnt
        lbl_str = ', '.join(f'{k}:{v//1000}k' for k,v in file_lbls.items())
        print(f'  {fname:<35} rows={file_rows:>10,}  BENIGN={benign_cnt:>8,}  attack={attack_cnt:>10,}')
        file_summary.append({'file': fname, 'rows': file_rows, 'benign': benign_cnt, 'attack': attack_cnt})
    except Exception as e:
        print(f'  ERROR {fname}: {e}')

grand_total  = sum(x['rows']    for x in file_summary)
grand_benign = sum(x['benign']  for x in file_summary)
grand_attack = sum(x['attack']  for x in file_summary)
print(f'\nTOTAL: {grand_total:,} rows | BENIGN={grand_benign:,} | Attack={grand_attack:,}')

Found 18 CSV files
Total size: 31.06 GB

  DrDoS_DNS.csv                       rows= 5,074,413  BENIGN=   3,402  attack= 5,071,011
  DrDoS_LDAP.csv                      rows= 2,181,542  BENIGN=   1,612  attack= 2,179,930
  DrDoS_MSSQL.csv                     rows= 4,524,498  BENIGN=   2,006  attack= 4,522,492
  DrDoS_NTP.csv                       rows= 1,217,007  BENIGN=  14,365  attack= 1,202,642
  DrDoS_NetBIOS.csv                   rows= 4,094,986  BENIGN=   1,707  attack= 4,093,279
  DrDoS_SNMP.csv                      rows= 5,161,377  BENIGN=   1,507  attack= 5,159,870
  DrDoS_SSDP.csv                      rows= 2,611,374  BENIGN=     763  attack= 2,610,611
  DrDoS_UDP.csv                       rows= 3,136,802  BENIGN=   2,157  attack= 3,134,645
  Syn.csv                             rows= 1,582,681  BENIGN=     392  attack= 1,582,289
  TFTP.csv                            rows=20,107,827  BENIGN=  25,247  attack=20,082,580
  UDPLag.csv                          rows=   370,605  BENI

In [4]:
# ── Cell 3: Class distribution table ──────────────────────────────────────────
class_counts = defaultdict(int)
for lbl, cnt in all_labels.items():
    cls = LABEL_MAP.get(lbl, -1)
    if cls >= 0:
        class_counts[cls] += cnt

print('=== CLASS DISTRIBUTION (spec mapping) ===')
print(f'{"Class":<5} {"Name":<18} {"Rows":>14} {"Pct":>7}  {"Est. Windows (step=50)":>22}')
print('-' * 75)
for cls_id in range(7):
    cnt  = class_counts.get(cls_id, 0)
    pct  = 100 * cnt / max(grand_total, 1)
    wins = max(0, (cnt - 100) // 50) if cnt >= 100 else 0
    print(f'[{cls_id}]   {CLASS_NAMES[cls_id]:<18} {cnt:>14,} {pct:>6.2f}%  {wins:>22,}')
print('-' * 75)
print(f"{''  :5} {'TOTAL':<18} {grand_total:>14,} {100:>6.2f}%")

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
names = [CLASS_NAMES[i] for i in range(7)]
counts = [class_counts.get(i, 0) for i in range(7)]
colors = ['#2196F3','#F44336','#FF9800','#4CAF50','#9C27B0','#00BCD4','#795548']

axes[0].barh(names, counts, color=colors)
axes[0].set_title('Row Count per Class (raw)')
axes[0].set_xlabel('Number of rows')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1e6:.1f}M'))

non_zero = [(n, c) for n, c in zip(names, counts) if c > 0]
axes[1].pie([c for _,c in non_zero], labels=[n for n,_ in non_zero],
            colors=colors[:len(non_zero)], autopct='%1.1f%%', startangle=140)
axes[1].set_title('Class Distribution (row %)')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '01_class_distribution_raw.png', dpi=150)
plt.show()
print(f'Saved: {FIGURES_DIR}/01_class_distribution_raw.png')

=== CLASS DISTRIBUTION (spec mapping) ===
Class Name                         Rows     Pct  Est. Windows (step=50)
---------------------------------------------------------------------------
[0]   Normal                    113,828   0.16%                   2,274
[1]   UDP_Flood               7,370,134  10.46%                 147,400
[2]   SYN_Flood               6,473,789   9.19%                 129,473
[3]   HTTP_Flood                      0   0.00%                       0
[4]   ICMP_Flood                      0   0.00%                       0
[5]   Amplification          56,469,447  80.18%               1,129,386
[6]   Slow_rate                       0   0.00%                       0
---------------------------------------------------------------------------
      TOTAL                  70,427,637 100.00%
Saved: /kaggle/working/pad_onap_v3/figures/01_class_distribution_raw.png


---
## Section 2 — Feature Extraction (Streaming, Full Dataset)

In [5]:
# ── Cell 4: Feature extraction helpers ───────────────────────────────────────
def shannon_entropy(values: np.ndarray) -> float:
    if len(values) == 0: return 0.0
    _, counts = np.unique(values, return_counts=True)
    probs = counts / counts.sum()
    return float(-np.sum(probs * np.log2(probs + 1e-12)))


def extract_17_features(df_window: pd.DataFrame) -> np.ndarray:
    """Compute 17 spec features from a window of flow records."""
    n = len(df_window)
    if n == 0:
        return np.zeros(17, dtype=np.float32)

    def col(name, default=0.0):
        return df_window[name].values if name in df_window.columns else np.full(n, default)

    dur = col('Flow Duration', 1.0)
    dur = np.where(dur <= 0, 1.0, dur)

    total_pkts  = col('Total Fwd Packets') + col('Total Backward Packets')
    total_bytes = col('Total Length of Fwd Packets') + col('Total Length of Bwd Packets')

    pkt_rate  = float(np.sum(total_pkts)  / np.sum(dur) * 1e6)
    byte_rate = float(np.sum(total_bytes) / np.sum(dur) * 1e6)

    src_port_entropy = shannon_entropy(col('Source Port',      0).astype(int))
    dst_port_entropy = shannon_entropy(col('Destination Port', 0).astype(int))

    if 'Source IP' in df_window.columns:
        src_ip_entropy = shannon_entropy(df_window['Source IP'].astype(str).values)
    else:
        # FIX: if IP columns are missing, avoid synthetic proxy entropy.
        src_ip_entropy = 0.0

    if 'Destination IP' in df_window.columns:
        dst_ip_entropy = shannon_entropy(df_window['Destination IP'].astype(str).values)
    else:
        # FIX: if IP columns are missing, avoid synthetic proxy entropy.
        dst_ip_entropy = 0.0

    proto = col('Protocol', 6).astype(int)
    proto_dist_tcp  = float((proto == 6).sum()  / max(n, 1))
    proto_dist_udp  = float((proto == 17).sum() / max(n, 1))
    proto_dist_icmp = float((proto == 1).sum()  / max(n, 1))

    tcp_mask = (proto == 6)
    tcp_pkts = float(total_pkts[tcp_mask].sum()) if tcp_mask.any() else 1.0
    syn_ratio = float(col('SYN Flag Count', 0).sum() / max(tcp_pkts, 1))
    fin_ratio = float(col('FIN Flag Count', 0).sum() / max(tcp_pkts, 1))

    pkt_sizes   = col('Average Packet Size', 64.0)
    avg_pkt_size = float(pkt_sizes.mean())
    pkt_size_std = float(pkt_sizes.std() + 1e-8)

    win_dur_s    = float(np.sum(dur) / 1e6 )
    new_flows_rate  = float(n / max(win_dur_s, 0.001))
    flow_dur_mean   = float(dur.mean() / 1000.0)
    # L3 fix: Flow IAT is in microseconds → divide by 1e3 gives ms (consistent units)
    inter_arr_mean  = float(col('Flow IAT Mean', 1000.0).mean() / 1e3)   # ms
    inter_arr_std   = float(col('Flow IAT Std',   500.0).mean() / 1e3)   # ms

    feat = np.array([
        pkt_rate, byte_rate,
        src_ip_entropy, dst_ip_entropy,
        src_port_entropy, dst_port_entropy,
        proto_dist_tcp, proto_dist_udp, proto_dist_icmp,
        syn_ratio, fin_ratio,
        avg_pkt_size, pkt_size_std,
        new_flows_rate, flow_dur_mean,
        inter_arr_mean, inter_arr_std,
    ], dtype=np.float32)
    return np.nan_to_num(feat, nan=0.0, posinf=1e6, neginf=0.0)

print('Feature helpers ready.')

Feature helpers ready.


In [6]:
# ── Cell 5: Streaming mixed-temporal feature extraction (class/file quota) ─────
# Idea: keep MAX_WINDOWS but prevent early-file domination by combining:
#   - Class quota (minimum per class + proportional remainder)
#   - Dynamic file quota (balanced contribution among remaining files)
#   - Safe overflow when strict class quota would underfill MAX_WINDOWS

WINDOW_SIZE = 100
STEP = 50
MAX_WINDOWS = 240_000  # keep train split under Transformer MAX_TRAIN cap
MIN_CLASS_WINDOWS = 20_000

mixed_windows_X = []
mixed_windows_y = []
mixed_segment_lengths = []
total_rows_read = 0
unknown_rows = 0
t_start = time.time()


def window_majority_label(labels: np.ndarray) -> int:
    # Majority vote with tie-break on latest row to better preserve event onset.
    counts = np.bincount(labels, minlength=7)
    top = counts.max()
    candidates = np.where(counts == top)[0]
    if len(candidates) == 1:
        return int(candidates[0])
    return int(labels[-1])


def build_class_quota(max_windows: int, min_per_class: int):
    # Use audited row distribution to compute active classes and weighted quotas.
    class_rows = {i: int(class_counts.get(i, 0)) for i in range(7)}
    active = [c for c, n in class_rows.items() if n > 0]
    if not active:
        active = sorted({int(v) for v in LABEL_MAP.values() if int(v) >= 0})

    quota = {c: 0 for c in range(7)}
    k = len(active)
    if k == 0:
        return quota, []

    min_total = min_per_class * k
    if min_total >= max_windows:
        base = max_windows // k
        rem = max_windows - base * k
        for i, c in enumerate(active):
            quota[c] = base + (1 if i < rem else 0)
        return quota, active

    for c in active:
        quota[c] = min_per_class

    remain = max_windows - min_total
    weights = np.array([max(class_rows[c], 1) for c in active], dtype=np.float64)
    weights = weights / weights.sum()
    raw = remain * weights
    floor_add = np.floor(raw).astype(int)
    frac = raw - floor_add

    for i, c in enumerate(active):
        quota[c] += int(floor_add[i])

    rem = int(remain - floor_add.sum())
    order = np.argsort(-frac)
    for i in range(rem):
        quota[active[order[i % k]]] += 1

    return quota, active


class_quota, active_classes = build_class_quota(MAX_WINDOWS, MIN_CLASS_WINDOWS)
class_windows = defaultdict(int)

print(f'Extracting mixed temporal windows from {len(csv_files)} CSV files...')
print(f'WINDOW_SIZE={WINDOW_SIZE}, STEP={STEP}, MAX_WINDOWS={MAX_WINDOWS}')
print(f'Active classes: {[CLASS_NAMES.get(c, c) for c in active_classes]}')
print('Class quota plan:')
for c in active_classes:
    print(f'  - {CLASS_NAMES.get(c, c):<15}: {class_quota[c]:>7,} windows')
print()

for fi, f in enumerate(csv_files):
    if len(mixed_windows_X) >= MAX_WINDOWS:
        break

    # Dynamic file quota: each remaining file gets a fair share of remaining slots.
    remaining_slots = MAX_WINDOWS - len(mixed_windows_X)
    remaining_files = max(1, len(csv_files) - fi)
    file_quota = int(np.ceil(remaining_slots / remaining_files))

    fname = os.path.basename(f)
    file_rows = 0
    file_windows = 0
    file_windows_by_class = defaultdict(int)
    carry_over = pd.DataFrame()  # reset per-file to avoid synthetic cross-file windows

    try:
        for chunk in pd.read_csv(f, chunksize=200_000, low_memory=False, on_bad_lines='skip'):
            chunk.columns = [c.strip() for c in chunk.columns]
            lcol = next((c for c in chunk.columns if c.lower() == 'label'), None)
            if lcol is None:
                continue

            mapped = chunk[lcol].str.strip().map(LABEL_MAP)
            unknown_rows += int(mapped.isna().sum())
            chunk['_class'] = mapped
            chunk = chunk.dropna(subset=['_class'])
            if len(chunk) == 0:
                continue
            chunk['_class'] = chunk['_class'].astype(int)

            file_rows += len(chunk)
            total_rows_read += len(chunk)

            df_combined = (
                pd.concat([carry_over, chunk], ignore_index=True)
                if len(carry_over) > 0 else chunk.reset_index(drop=True)
            )

            i = 0
            while i + WINDOW_SIZE <= len(df_combined):
                if len(mixed_windows_X) >= MAX_WINDOWS or file_windows >= file_quota:
                    break

                window = df_combined.iloc[i:i + WINDOW_SIZE]
                feats = extract_17_features(window)
                lbl = window_majority_label(window['_class'].values.astype(np.int32))

                # Enforce class quota, but allow overflow if strict quotas would underfill MAX_WINDOWS.
                remaining_quota = sum(max(0, class_quota[c] - class_windows[c]) for c in active_classes)
                remaining_need = MAX_WINDOWS - len(mixed_windows_X)
                class_allowed = (
                    class_windows[lbl] < class_quota.get(lbl, 0)
                    or remaining_quota < remaining_need
                )

                if class_allowed:
                    mixed_windows_X.append(feats)
                    mixed_windows_y.append(lbl)
                    class_windows[lbl] += 1
                    file_windows_by_class[lbl] += 1
                    file_windows += 1

                i += STEP

            carry_over = df_combined.iloc[i:].reset_index(drop=True)
            if len(mixed_windows_X) >= MAX_WINDOWS or file_windows >= file_quota:
                break

        if file_windows > 0:
            mixed_segment_lengths.append(file_windows)

        elapsed = time.time() - t_start
        if file_windows > 0:
            parts = [
                f"{CLASS_NAMES.get(c, 'Unknown')}: {cnt}"
                for c, cnt in sorted(file_windows_by_class.items())
            ]
            class_dist_str = ' | '.join(parts)
        else:
            class_dist_str = '0 windows'

        print(
            f'  [{fi+1:2d}/{len(csv_files)}] {fname:<35} rows={file_rows:>8,}\n'
            f'       └─ FileQuota={file_quota:>6,} | Windows={file_windows:>7,} | Total={len(mixed_windows_X):>8,} | Time={elapsed:.0f}s\n'
            f'       └─ Classes: {class_dist_str}\n'
        )
    except Exception as e:
        print(f'  ERROR {fname}: {e}')

if len(mixed_windows_X) == 0:
    raise ValueError('No mixed windows extracted. Please check LABEL_MAP and input CSV files.')

X_mixed = np.stack(mixed_windows_X).astype(np.float32)
y_mixed = np.array(mixed_windows_y, dtype=np.int32)
mixed_segment_lengths = [int(x) for x in mixed_segment_lengths if x > 0]

final_classes, final_counts = np.unique(y_mixed, return_counts=True)
final_dist = {CLASS_NAMES.get(c, c): int(cnt) for c, cnt in zip(final_classes, final_counts)}

print('=' * 60)
print('EXTRACTION SUMMARY:')
print(f'Total rows read: {total_rows_read:,}')
print(f'Unknown/unmapped rows dropped: {unknown_rows:,}')
print(f'Extracted windows: {len(X_mixed):,}')
print(f'Mixed segments (per file): {len(mixed_segment_lengths)}')
print(f'FINAL Class dist (mixed windows): {final_dist}')
print('Class quota usage:')
for c in active_classes:
    used = int(class_windows[c])
    q = int(class_quota[c])
    print(f'  - {CLASS_NAMES.get(c, c):<15}: {used:>7,} / {q:>7,}')
print('=' * 60)

Extracting mixed temporal windows from 18 CSV files...
WINDOW_SIZE=100, STEP=50, MAX_WINDOWS=240000
Active classes: ['Normal', 'UDP_Flood', 'SYN_Flood', 'Amplification']
Class quota plan:
  - Normal         :  20,259 windows
  - UDP_Flood      :  36,744 windows
  - SYN_Flood      :  34,707 windows
  - Amplification  : 148,290 windows

  [ 1/18] DrDoS_DNS.csv                       rows= 800,000
       └─ FileQuota=13,334 | Windows= 13,334 | Total=  13,334 | Time=28s
       └─ Classes: Normal: 19 | Amplification: 13315

  [ 2/18] DrDoS_LDAP.csv                      rows= 800,000
       └─ FileQuota=13,334 | Windows= 13,334 | Total=  26,668 | Time=56s
       └─ Classes: Amplification: 13334

  [ 3/18] DrDoS_MSSQL.csv                     rows= 800,000
       └─ FileQuota=13,334 | Windows= 13,334 | Total=  40,002 | Time=85s
       └─ Classes: Normal: 16 | Amplification: 13318

  [ 4/18] DrDoS_NTP.csv                       rows= 800,000
       └─ FileQuota=13,334 | Windows= 13,334 | Total=  

In [8]:
# ── Cell 6: Temporal split + compatible outputs for both tracks ───────────────
# Safe-merge behavior: keep mixed temporal idea, while preserving all variable
# names expected by downstream cells (X_train/X_test, X_tf_*, segment lengths).

TEMPORAL_SPLIT_RATIO = 0.8
PURGE_WINDOWS = 10
MAX_BENIGN_OVERSAMPLE = 20
rng_fix = np.random.default_rng(42)

if 'X_mixed' not in globals() or 'y_mixed' not in globals():
    raise ValueError('X_mixed/y_mixed not found. Run Cell 5 first.')

n_total = len(X_mixed)
split_idx = int(n_total * TEMPORAL_SPLIT_RATIO)
test_start = min(n_total, split_idx + PURGE_WINDOWS)

if split_idx <= 0 or test_start >= n_total:
    raise ValueError(f'Invalid temporal split: total={n_total}, split={split_idx}, test_start={test_start}')

# Base temporal split (unshuffled)
X_train_raw_base = X_mixed[:split_idx]
y_train_base = y_mixed[:split_idx]
X_test_raw_base = X_mixed[test_start:]
y_test_base = y_mixed[test_start:]

# Compatibility variables used by summary cell
total_benign_rows = int(grand_benign) if 'grand_benign' in globals() else int((y_mixed == 0).sum())
raw_benign_X = X_mixed[y_mixed == 0].astype(np.float32)
n_raw = len(raw_benign_X)

# Train balancing: capped BENIGN oversampling only on train split
bn_mask = (y_train_base == 0)
atk_mask = ~bn_mask
X_bn_tr_base = X_train_raw_base[bn_mask]
X_atk_tr_base = X_train_raw_base[atk_mask]
y_atk_tr_base = y_train_base[atk_mask]

if len(X_bn_tr_base) == 0 or len(X_atk_tr_base) == 0:
    raise ValueError(
        f'Train split has empty class group: benign={len(X_bn_tr_base)}, attack={len(X_atk_tr_base)}. '
        'Adjust split ratio or extraction settings.'
    )

target_bn_tr = min(len(X_atk_tr_base), len(X_bn_tr_base) * MAX_BENIGN_OVERSAMPLE)
bn_idx = rng_fix.choice(len(X_bn_tr_base), size=target_bn_tr, replace=(len(X_bn_tr_base) < target_bn_tr))
X_bn_tr = X_bn_tr_base[bn_idx]
y_bn_tr = np.zeros(len(X_bn_tr), dtype=np.int32)

X_train_raw = np.vstack([X_bn_tr, X_atk_tr_base]).astype(np.float32)
y_train = np.hstack([y_bn_tr, y_atk_tr_base]).astype(np.int32)

# Test set remains natural distribution (no BENIGN oversampling)
X_test_raw = X_test_raw_base.astype(np.float32)
y_test = y_test_base.astype(np.int32)

# Shuffle within split only (keeps temporal split integrity)
s1 = rng_fix.permutation(len(X_train_raw))
X_train_raw, y_train = X_train_raw[s1], y_train[s1]
s2 = rng_fix.permutation(len(X_test_raw))
X_test_raw, y_test = X_test_raw[s2], y_test[s2]

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
X_test = scaler.transform(X_test_raw).astype(np.float32)

print(f'Temporal split: train_base={len(X_train_raw_base):,} | purge={PURGE_WINDOWS} | test_base={len(X_test_raw_base):,}')
print(f'BENIGN train: {len(X_bn_tr_base):,} -> {len(X_bn_tr):,} (cap x{MAX_BENIGN_OVERSAMPLE})')
print('BENIGN test:  natural distribution (no oversampling)')
print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'Train class dist: {dict(zip(*np.unique(y_train, return_counts=True)))}')
print(f'Test  class dist: {dict(zip(*np.unique(y_test, return_counts=True)))}')

np.save(DATA_DIR / 'X_train.npy', X_train)
np.save(DATA_DIR / 'X_test.npy',  X_test)
np.save(DATA_DIR / 'y_train.npy', y_train)
np.save(DATA_DIR / 'y_test.npy',  y_test)
np.save(DATA_DIR / 'X_benign_raw.npy', raw_benign_X)
with open(DATA_DIR / 'scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Transformer temporal data (unshuffled base split, no extra resampling)
X_tf_train_raw = X_train_raw_base.astype(np.float32)
X_tf_test_raw = X_test_raw_base.astype(np.float32)
y_tf_train = y_train_base.astype(np.int32)
y_tf_test = y_test_base.astype(np.int32)

# Build segment lengths from file-wise mixed segments to prevent crossing boundaries.
tf_train_segment_lengths = []
tf_test_segment_lengths = []
cursor = 0
for seg_len in mixed_segment_lengths:
    seg_start = cursor
    seg_end = cursor + int(seg_len)

    tr_len = max(0, min(seg_end, split_idx) - seg_start)
    if tr_len > 0:
        tf_train_segment_lengths.append(int(tr_len))

    te_len = max(0, seg_end - max(seg_start, test_start))
    if te_len > 0:
        tf_test_segment_lengths.append(int(te_len))

    cursor = seg_end

if sum(tf_train_segment_lengths) != len(X_tf_train_raw):
    raise ValueError('tf_train_segment_lengths mismatch with X_tf_train_raw length')
if sum(tf_test_segment_lengths) != len(X_tf_test_raw):
    raise ValueError('tf_test_segment_lengths mismatch with X_tf_test_raw length')

X_tf_train = scaler.transform(X_tf_train_raw).astype(np.float32)
X_tf_test = scaler.transform(X_tf_test_raw).astype(np.float32)

print('Transformer temporal data:')
print(f'  X_tf_train: {X_tf_train.shape} | X_tf_test: {X_tf_test.shape}')
print(f'  tf_train segments: {tf_train_segment_lengths[:8]} ... total={len(tf_train_segment_lengths)}')
print(f'  tf_test  segments: {tf_test_segment_lengths[:8]} ... total={len(tf_test_segment_lengths)}')
print(f'  tf_train class dist: {dict(zip(*np.unique(y_tf_train, return_counts=True)))}')
print(f'  tf_test  class dist: {dict(zip(*np.unique(y_tf_test, return_counts=True)))}')

metadata = {
    'total_rows_read': int(total_rows_read),
    'total_benign_rows': int(total_benign_rows),
    'raw_benign_windows': int(n_raw),
    'window_size': int(WINDOW_SIZE),
    'step': int(STEP),
    'n_features': 17,
    'feature_names': FEATURE_NAMES,
    'split_strategy': 'mixed_temporal_split_with_purge',
    'temporal_split_ratio': float(TEMPORAL_SPLIT_RATIO),
    'purge_windows': int(PURGE_WINDOWS),
    'max_benign_oversample': int(MAX_BENIGN_OVERSAMPLE),
    'max_windows': int(MAX_WINDOWS),
    'train_samples': int(len(X_train)),
    'test_samples': int(len(X_test)),
    'class_dist_train': {CLASS_NAMES[i]: int((y_train == i).sum()) for i in range(7)},
    'class_dist_test': {CLASS_NAMES[i]: int((y_test == i).sum()) for i in range(7)},
    'tf_train_samples': int(len(X_tf_train)),
    'tf_test_samples': int(len(X_tf_test)),
    'tf_train_segments': [int(x) for x in tf_train_segment_lengths],
    'tf_test_segments': [int(x) for x in tf_test_segment_lengths],
}
with open(DATA_DIR / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'Saved to {DATA_DIR}')

Temporal split: train_base=176,880 | purge=10 | test_base=44,210
BENIGN train: 487 -> 9,740 (cap x20)
BENIGN test:  natural distribution (no oversampling)
X_train: (186133, 17) | X_test: (44210, 17)
Train class dist: {np.int32(0): np.int64(9740), np.int32(1): np.int64(20654), np.int32(2): np.int64(13333), np.int32(5): np.int64(142406)}
Test  class dist: {np.int32(0): np.int64(872), np.int32(1): np.int64(16090), np.int32(2): np.int64(21374), np.int32(5): np.int64(5874)}
Transformer temporal data:
  X_tf_train: (176880, 17) | X_tf_test: (44210, 17)
  tf_train segments: [13334, 13334, 13334, 13334, 13334, 13334, 13333, 13333] ... total=14
  tf_test  segments: [5874, 98, 19046, 16114, 3078] ... total=5
  tf_train class dist: {np.int32(0): np.int64(487), np.int32(1): np.int64(20654), np.int32(2): np.int64(13333), np.int32(5): np.int64(142406)}
  tf_test  class dist: {np.int32(0): np.int64(872), np.int32(1): np.int64(16090), np.int32(2): np.int64(21374), np.int32(5): np.int64(5874)}
Saved to

In [9]:
# ── Cell 7: Post-extraction statistics ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

train_counts = {CLASS_NAMES[i]: int((y_train==i).sum()) for i in range(7) if (y_train==i).sum()>0}
test_counts  = {CLASS_NAMES[i]: int((y_test==i).sum())  for i in range(7) if (y_test==i).sum()>0}

x_tr = list(train_counts.keys())
axes[0].bar(x_tr, list(train_counts.values()), color='steelblue')
axes[0].set_title('Train Set — Class Distribution')
axes[0].set_ylabel('Windows'); axes[0].tick_params(axis='x', rotation=30)

x_te = list(test_counts.keys())
axes[1].bar(x_te, list(test_counts.values()), color='coral')
axes[1].set_title('Test Set — Class Distribution')
axes[1].set_ylabel('Windows'); axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_train_test_dist.png', dpi=150)
plt.show()

print(f'Train: {X_train.shape}')
print(f'Test:  {X_test.shape}')
print(f'\nFeature stats (train):')
df_stats = pd.DataFrame(X_train, columns=FEATURE_NAMES).describe().T
display(df_stats[['mean','std','min','max']].round(4))

Train: (186133, 17)
Test:  (44210, 17)

Feature stats (train):


,mean,std,min,max
pkt_rate,-0.0,1.0002,-0.7488,6.0314
byte_rate,0.0,0.9995,-0.5340,6.5276
src_ip_entropy,0.0,0.9996,-0.2301,7.7008
dst_ip_entropy,0.0,1.0007,-0.2439,6.4169
src_port_entropy,-0.0,0.9997,-6.1603,0.4744
dst_port_entropy,-0.0,1.0006,-6.7546,0.2672
proto_dist_tcp,-0.0,1.0007,-0.4473,2.4441
proto_dist_udp,-0.0,1.0007,-2.4384,0.4478
proto_dist_icmp,0.0,0.0000,0.0000,0.0000
syn_ratio,0.0,0.9997,-0.0262,72.1553


---
## Section 3 — Track A: XGBoost 7-class Classifier

In [10]:
# ── Cell 8: Gaussian noise augmentation (regularisation) ──────────────────────
# L5 fix: original code used np.gradient (finite-diff between rows) and called
# it FGSM — that is NOT model-gradient-based FGSM. Replaced with honest
# Gaussian noise augmentation which achieves the same regularisation goal.
def gaussian_noise_augment(X: np.ndarray, sigma: float = 0.01) -> np.ndarray:
    """Add Gaussian noise scaled by per-feature std for regularisation."""
    feat_std = X.std(axis=0, keepdims=True).clip(1e-6)
    noise    = np.random.default_rng(7).normal(0, sigma, X.shape).astype(np.float32)
    X_noisy  = X + noise * feat_std
    return np.clip(X_noisy, X.min(axis=0), X.max(axis=0))

def augment_with_noise(X: np.ndarray, y: np.ndarray, sigma: float = 0.01):
    mask  = y != 0                        # augment only attack samples
    X_aug_part = gaussian_noise_augment(X[mask], sigma)
    X_aug = np.vstack([X, X_aug_part])
    y_aug = np.hstack([y, y[mask]])
    print(f'Noise augment: +{X_aug_part.shape[0]:,} samples → total {len(X_aug):,}')
    return X_aug, y_aug

X_train_aug, y_train_aug = augment_with_noise(X_train, y_train, sigma=0.01)

Noise augment: +176,393 samples → total 362,526


In [ ]:
# ── Cell 8.5: Optuna tuning for XGBoost (run-once/reuse) ─────────────────────
import optuna
from sklearn.model_selection import train_test_split

N_TRIALS_XGB = 20
TIMEOUT_XGB_SEC = 1200
TOP_K_XGB = 3

# Workflow controls (override via Cell 1.5 when present)
RUN_XGB_TUNING = globals().get('RUN_XGB_TUNING', True)
LOAD_XGB_TUNED = globals().get('LOAD_XGB_TUNED', True)
SAVE_XGB_TUNED = globals().get('SAVE_XGB_TUNED', True)
XGB_TUNED_PATH = MODELS_DIR / 'xgb_tuned_configs.json'

loaded_ok = False
if LOAD_XGB_TUNED and XGB_TUNED_PATH.exists():
    try:
        with open(XGB_TUNED_PATH, 'r') as f:
            loaded = json.load(f)
        if isinstance(loaded, dict) and 'configs' in loaded and len(loaded['configs']) > 0:
            XGB_HP_CONFIGS_OVERRIDE = loaded['configs']
            loaded_ok = True
            print(f'Loaded tuned XGB configs from {XGB_TUNED_PATH}')
            print(f"  source_time={loaded.get('saved_at','unknown')} | best_macro_f1={loaded.get('best_value','n/a')}")
            for i, hp in enumerate(XGB_HP_CONFIGS_OVERRIDE, start=1):
                print(f'  {i}. {hp}')
    except Exception as e:
        print(f'Failed to load tuned XGB configs: {e}')

if (not loaded_ok) and RUN_XGB_TUNING:
    # Use augmented train only; keep test untouched for final evaluation.
    X_tune_tr, X_tune_val, y_tune_tr, y_tune_val = train_test_split(
        X_train_aug, y_train_aug, test_size=0.2, stratify=y_train_aug, random_state=42
    )

    print(f'XGB tuning split: train={X_tune_tr.shape}, val={X_tune_val.shape}')

    def _fit_eval_xgb_trial(trial):
        hp = {
            'n_estimators': trial.suggest_int('n_estimators', 200, 700, step=100),
            'max_depth': trial.suggest_int('max_depth', 6, 12),
            'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.12, log=True),
            'subsample': trial.suggest_float('subsample', 0.7, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
            'min_child_weight': trial.suggest_float('min_child_weight', 1.0, 12.0),
            'gamma': trial.suggest_float('gamma', 0.0, 3.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 2.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        }

        labels = sorted(np.unique(np.concatenate([y_tune_tr, y_tune_val])))
        lbl2idx = {l: i for i, l in enumerate(labels)}
        idx2lbl = {i: l for l, i in lbl2idx.items()}

        ytr = np.array([lbl2idx[l] for l in y_tune_tr], dtype=np.int32)
        yva = np.array([lbl2idx[l] for l in y_tune_val], dtype=np.int32)
        n_cls = len(labels)

        cls_counts = np.bincount(ytr, minlength=n_cls).astype(np.float32)
        cls_counts = np.clip(cls_counts, 1.0, None)
        cls_weights = cls_counts.sum() / (n_cls * cls_counts)
        wtr = cls_weights[ytr]

        dtr = xgb.DMatrix(X_tune_tr, label=ytr, weight=wtr)
        dva = xgb.DMatrix(X_tune_val, label=yva)

        params = {
            'device': XGB_DEVICE, 'tree_method': 'hist',
            'objective': 'multi:softprob', 'num_class': n_cls,
            'eval_metric': 'mlogloss', 'seed': 42, 'verbosity': 0,
            'max_depth': hp['max_depth'],
            'learning_rate': hp['learning_rate'],
            'subsample': hp['subsample'],
            'colsample_bytree': hp['colsample_bytree'],
            'min_child_weight': hp['min_child_weight'],
            'gamma': hp['gamma'],
            'reg_alpha': hp['reg_alpha'],
            'reg_lambda': hp['reg_lambda'],
        }

        booster = xgb.train(
            params, dtr,
            num_boost_round=hp['n_estimators'],
            evals=[(dva, 'val')],
            callbacks=[xgb.callback.EarlyStopping(rounds=30, save_best=True)],
            verbose_eval=False,
        )

        raw = booster.predict(dva)
        yproba = raw.reshape(-1, n_cls) if raw.ndim == 1 else raw
        ypred = np.argmax(yproba, axis=1)
        ypred_lbl = np.array([idx2lbl[i] for i in ypred], dtype=np.int32)

        score = f1_score(
            y_tune_val, ypred_lbl,
            labels=sorted(set(np.unique(y_tune_tr).tolist()) | set(np.unique(y_tune_val).tolist())),
            average='macro', zero_division=0
        )
        return float(score)

    study = optuna.create_study(direction='maximize', study_name='xgb_macro_f1')
    study.optimize(_fit_eval_xgb_trial, n_trials=N_TRIALS_XGB, timeout=TIMEOUT_XGB_SEC, show_progress_bar=False)

    top_trials = sorted(study.trials, key=lambda t: t.value if t.value is not None else -1, reverse=True)[:TOP_K_XGB]
    XGB_HP_CONFIGS_OVERRIDE = []
    for t in top_trials:
        p = t.params
        XGB_HP_CONFIGS_OVERRIDE.append({
            'n_estimators': int(p['n_estimators']),
            'max_depth': int(p['max_depth']),
            'learning_rate': float(p['learning_rate']),
            'subsample': float(p['subsample']),
            'colsample_bytree': float(p['colsample_bytree']),
            'min_child_weight': float(p['min_child_weight']),
            'gamma': float(p['gamma']),
            'reg_alpha': float(p['reg_alpha']),
            'reg_lambda': float(p['reg_lambda']),
        })

    print(f'Best XGB trial macro_f1={study.best_value:.4f}')
    print('Using tuned configs (top-k):')
    for i, hp in enumerate(XGB_HP_CONFIGS_OVERRIDE, start=1):
        print(f'  {i}. {hp}')

    if SAVE_XGB_TUNED:
        payload = {
            'saved_at': time.strftime('%Y-%m-%d %H:%M:%S'),
            'best_value': float(study.best_value),
            'n_trials': int(len(study.trials)),
            'configs': XGB_HP_CONFIGS_OVERRIDE,
        }
        with open(XGB_TUNED_PATH, 'w') as f:
            json.dump(payload, f, indent=2)
        print(f'Saved tuned XGB configs to {XGB_TUNED_PATH}')

if 'XGB_HP_CONFIGS_OVERRIDE' not in globals():
    print('No tuned XGB override loaded/generated. Cell 9 will use default configs.')

In [ ]:
# ── Cell 9: XGBoost training ──────────────────────────────────────────────────
TARGET_XGB = {'accuracy': 0.95, 'macro_f1': 0.90, 'balanced_acc': 0.90}

XGB_HP_CONFIGS = [
    dict(n_estimators=300, max_depth=8,  learning_rate=0.05, subsample=0.8, colsample_bytree=0.8),
    dict(n_estimators=500, max_depth=8,  learning_rate=0.05, subsample=0.9, colsample_bytree=0.9),
    dict(n_estimators=500, max_depth=10, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8),
    dict(n_estimators=700, max_depth=10, learning_rate=0.03, subsample=0.9, colsample_bytree=0.8),
]

if 'XGB_HP_CONFIGS_OVERRIDE' in globals() and len(XGB_HP_CONFIGS_OVERRIDE) > 0:
    XGB_HP_CONFIGS = XGB_HP_CONFIGS_OVERRIDE
    print(f'Using tuned XGB configs from Optuna: {len(XGB_HP_CONFIGS)} trial(s).')


def train_xgboost(X_tr, y_tr, X_te, y_te, hp, device):
    all_labels = sorted(np.unique(np.concatenate([y_tr, y_te])))
    n_cls = len(all_labels)
    lbl2idx = {l: i for i, l in enumerate(all_labels)}
    idx2lbl = {i: l for l, i in lbl2idx.items()}

    y_tr_r = np.array([lbl2idx[l] for l in y_tr], dtype=np.int32)
    y_te_r = np.array([lbl2idx[l] for l in y_te], dtype=np.int32)

    cls_counts = np.bincount(y_tr_r, minlength=n_cls).astype(np.float32)
    cls_counts = np.clip(cls_counts, 1.0, None)
    cls_weights = cls_counts.sum() / (n_cls * cls_counts)
    w_tr = cls_weights[y_tr_r]

    dtrain = xgb.DMatrix(X_tr, label=y_tr_r, weight=w_tr)
    dtest = xgb.DMatrix(X_te, label=y_te_r)

    params = {
        'device': device, 'tree_method': 'hist',
        'objective': 'multi:softprob', 'num_class': n_cls,
        'eval_metric': 'mlogloss', 'seed': 42, 'verbosity': 0,
        'max_depth':        hp.get('max_depth',        8),
        'learning_rate':    hp.get('learning_rate',    0.05),
        'subsample':        hp.get('subsample',        0.8),
        'colsample_bytree': hp.get('colsample_bytree', 0.8),
        'min_child_weight': hp.get('min_child_weight', 1.0),
        'gamma':            hp.get('gamma',            0.0),
        'reg_alpha':        hp.get('reg_alpha',        0.0),
        'reg_lambda':       hp.get('reg_lambda',       1.0),
    }
    t0 = time.time()
    booster = xgb.train(
        params, dtrain,
        num_boost_round=hp.get('n_estimators', 300),
        evals=[(dtest, 'eval')],
        callbacks=[xgb.callback.EarlyStopping(rounds=30, save_best=True)],
        verbose_eval=100,
    )
    elapsed = time.time() - t0

    raw = booster.predict(dtest)
    y_proba_r = raw.reshape(-1, n_cls) if raw.ndim == 1 else raw
    y_pred_r = np.argmax(y_proba_r, axis=1)
    y_pred = np.array([idx2lbl[i] for i in y_pred_r], dtype=np.int32)

    y_proba = np.zeros((len(y_te), 7), dtype=np.float32)
    for ri, ol in idx2lbl.items():
        if ol < 7:
            y_proba[:, ol] = y_proba_r[:, ri]

    acc = float(accuracy_score(y_te, y_pred))
    bal_acc = float(balanced_accuracy_score(y_te, y_pred))
    all_classes = sorted(set(np.unique(y_tr).tolist()) | set(np.unique(y_te).tolist()))
    macro_f1 = float(f1_score(y_te, y_pred, labels=all_classes, average='macro', zero_division=0))

    classes_present = np.unique(y_te).tolist()
    try:
        y_bin = label_binarize(y_te, classes=list(range(7)))
        auc = float(roc_auc_score(
            y_bin[:, classes_present], y_proba[:, classes_present],
            multi_class='ovr', average='macro'
        ))
    except Exception:
        auc = 0.0

    booster._label_to_idx = lbl2idx
    booster._idx_to_label = idx2lbl
    booster._n_classes = n_cls
    metrics = {
        'accuracy': acc,
        'balanced_acc': bal_acc,
        'macro_f1': macro_f1,
        'auc_macro_ovr': auc,
    }
    return booster, metrics, elapsed, y_pred, y_proba


best_xgb, best_xgb_metrics, best_y_pred, best_y_proba = None, {}, None, None

for attempt, hp in enumerate(XGB_HP_CONFIGS):
    print(f'\n--- Attempt {attempt+1}/{len(XGB_HP_CONFIGS)}: {hp} ---')
    booster, metrics, elapsed, y_pred, y_proba = train_xgboost(
        X_train_aug, y_train_aug, X_test, y_test, hp, XGB_DEVICE
    )
    print(
        f'  accuracy={metrics["accuracy"]:.4f}  balanced_acc={metrics["balanced_acc"]:.4f}  '
        f'macro_f1={metrics["macro_f1"]:.4f}  auc={metrics["auc_macro_ovr"]:.4f}  time={elapsed:.1f}s'
    )

    if not best_xgb_metrics or metrics['macro_f1'] > best_xgb_metrics.get('macro_f1', 0):
        best_xgb, best_xgb_metrics, best_y_pred, best_y_proba = booster, metrics, y_pred, y_proba

    if all(metrics.get(k, 0) >= v for k, v in TARGET_XGB.items()):
        print('All targets met!')
        break
    else:
        missing = [k for k, v in TARGET_XGB.items() if metrics.get(k, 0) < v]
        print(f'  Missing: {missing} - trying next config...')

print(
    f'\nBEST XGBoost: accuracy={best_xgb_metrics["accuracy"]:.4f}  '
    f'balanced_acc={best_xgb_metrics["balanced_acc"]:.4f}  '
    f'macro_f1={best_xgb_metrics["macro_f1"]:.4f}  auc={best_xgb_metrics["auc_macro_ovr"]:.4f}'
)


--- Attempt 1/4: {'n_estimators': 300, 'max_depth': 8, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8} ---
[0]	eval-mlogloss:1.32380
[100]	eval-mlogloss:0.26326
[137]	eval-mlogloss:0.26511
  accuracy=0.9062  balanced_acc=0.9501  macro_f1=0.9365  auc=0.9948  time=2.6s
  Missing: ['accuracy'] - trying next config...

--- Attempt 2/4: {'n_estimators': 500, 'max_depth': 8, 'learning_rate': 0.05, 'subsample': 0.9, 'colsample_bytree': 0.9} ---
[0]	eval-mlogloss:1.32150
[100]	eval-mlogloss:0.28101
[123]	eval-mlogloss:0.28947
  accuracy=0.9090  balanced_acc=0.9516  macro_f1=0.9384  auc=0.9940  time=2.0s
  Missing: ['accuracy'] - trying next config...

--- Attempt 3/4: {'n_estimators': 500, 'max_depth': 10, 'learning_rate': 0.03, 'subsample': 0.8, 'colsample_bytree': 0.8} ---
[0]	eval-mlogloss:1.34900
[100]	eval-mlogloss:0.32299
[200]	eval-mlogloss:0.30217
[208]	eval-mlogloss:0.30348
  accuracy=0.9073  balanced_acc=0.9511  macro_f1=0.9367  auc=0.9901  time=4.5s
  Missing: ['a

In [12]:
# ── Cell 10: XGBoost full eval + confusion matrix + SHAP ─────────────────────
present_classes = sorted(np.unique(np.concatenate([y_train, y_test])))
target_names = [CLASS_NAMES[i] for i in present_classes]

print('=== CLASSIFICATION REPORT ===')
print(classification_report(
    y_test, best_y_pred,
    labels=present_classes, target_names=target_names,
    digits=4, zero_division=0
))

# PR-AUC (one-vs-rest) gives better signal on imbalanced setups
y_test_bin = label_binarize(y_test, classes=list(range(7)))
pr_auc_per_class = {}
for cls in present_classes:
    try:
        pr_auc_per_class[cls] = float(average_precision_score(y_test_bin[:, cls], best_y_proba[:, cls]))
    except Exception:
        pr_auc_per_class[cls] = 0.0

print('\n=== PR-AUC (OvR) ===')
for cls in present_classes:
    print(f'  {CLASS_NAMES[cls]:<15}: {pr_auc_per_class[cls]:.4f}')
print(f"  Macro PR-AUC     : {np.mean([pr_auc_per_class[c] for c in present_classes]):.4f}")

# Confusion Matrix
cm = confusion_matrix(y_test, best_y_pred, labels=present_classes)
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.colorbar(im, ax=ax)
ax.set(
    xticks=range(len(present_classes)), yticks=range(len(present_classes)),
    xticklabels=target_names, yticklabels=target_names,
    title='XGBoost Confusion Matrix', ylabel='True', xlabel='Predicted'
    )
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
thresh = cm.max() / 2
for i in range(len(present_classes)):
    for j in range(len(present_classes)):
        ax.text(j, i, format(cm[i, j], 'd'), ha='center', va='center',
                color='white' if cm[i, j] > thresh else 'black', fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '03_xgb_confusion_matrix.png', dpi=150)
plt.show()

# SHAP
print('\nComputing SHAP values (sample=500)...')
try:
    explainer = shap.TreeExplainer(best_xgb)
    X_shap = X_test[:500]
    shap_out = explainer(X_shap)
    sv = shap_out.values
    if sv.ndim == 3:
        mean_shap = np.abs(sv).mean(axis=(0, 2))
        shap_values = [sv[:, :, c] for c in range(sv.shape[2])]
    else:
        mean_shap = np.abs(sv).mean(axis=0)
        shap_values = [sv]

    top5_idx = np.argsort(mean_shap)[-5:][::-1]
    print('Top-5 SHAP features:')
    for i in top5_idx:
        print(f'  {FEATURE_NAMES[i]}: {mean_shap[i]:.4f}')

    plt.figure(figsize=(10, 6))
    shap_class_names = [CLASS_NAMES.get(best_xgb._idx_to_label[i], f'class_{i}')
                        for i in range(best_xgb._n_classes)]
    shap.summary_plot(
        shap_values, X_shap, feature_names=FEATURE_NAMES,
        class_names=shap_class_names, show=False, max_display=17
    )
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '04_shap_summary.png', dpi=150)
    plt.show()
except Exception as e:
    print(f'SHAP error: {e}')

# Save XGBoost model
xgb_path = MODELS_DIR / 'xgboost_v3.json'
best_xgb.save_model(str(xgb_path))
with open(MODELS_DIR / 'xgb_label_map.json', 'w') as f:
    json.dump({
        'label_to_idx': {str(int(k)): int(v) for k, v in best_xgb._label_to_idx.items()},
        'idx_to_label': {str(int(k)): int(v) for k, v in best_xgb._idx_to_label.items()},
        'n_classes': int(best_xgb._n_classes),
        'metrics': {
            k: (float(v) if hasattr(v, '__float__') else
                [float(x) for x in v] if isinstance(v, list) else v)
            for k, v in best_xgb_metrics.items()
        }
    }, f, indent=2)
print(f'\nSaved: {xgb_path}')

=== CLASSIFICATION REPORT ===
               precision    recall  f1-score   support

       Normal     0.9602    0.9966    0.9781       872
    UDP_Flood     0.8069    0.9914    0.8897     16090
    SYN_Flood     1.0000    0.8184    0.9001     21374
Amplification     0.9720    1.0000    0.9858      5874

     accuracy                         0.9090     44210
    macro avg     0.9348    0.9516    0.9384     44210
 weighted avg     0.9252    0.9090    0.9092     44210


=== PR-AUC (OvR) ===
  Normal         : 0.9961
  UDP_Flood      : 0.8971
  SYN_Flood      : 0.9992
  Amplification  : 1.0000
  Macro PR-AUC     : 0.9731

Computing SHAP values (sample=500)...
Top-5 SHAP features:
  proto_dist_udp: 0.4341
  dst_port_entropy: 0.3975
  proto_dist_tcp: 0.3863
  flow_duration_mean: 0.3590
  avg_pkt_size: 0.3041

Saved: /kaggle/working/pad_onap_v3/models/xgboost_v3.json


---
## Section 4 — Track B: Transformer+LSTM 4-Horizon Forecaster

In [13]:
# ── Cell 11: Model definition ─────────────────────────────────────────────────
N_TIMESTEPS = 12
N_FEATURES  = 17
N_HORIZONS  = 4
# BUG-6 FIX: horizon labels are window offsets, not guaranteed wall-clock seconds.
HORIZON_LABELS = ['+6w', '+12w', '+18w', '+24w']
PROACTIVE_THRESHOLD = 0.70

class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class TransformerLSTMForecaster(nn.Module):
    def __init__(self, input_dim=N_FEATURES, hidden_dim=64, num_heads=4, num_layers=2,
                 lstm_hidden=128, lstm_layers=2, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.pos_enc    = SinusoidalPE(hidden_dim)
        enc_layer       = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout, batch_first=True, activation='gelu')
        self.transformer = nn.TransformerEncoder(
            enc_layer, num_layers=num_layers, norm=nn.LayerNorm(hidden_dim))
        self.lstm = nn.LSTM(
            input_size=hidden_dim, hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            dropout=dropout if lstm_layers > 1 else 0, batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(lstm_hidden, 64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, N_HORIZONS))

    def forward(self, x):
        h = self.pos_enc(self.input_proj(x))
        h = self.transformer(h)
        h, _ = self.lstm(h)
        return self.fc(h[:, -1, :])


class ForecastDataset(Dataset):
    def __init__(self, X, y_class, n_timesteps=N_TIMESTEPS, segment_lengths=None):
        self.windows, self.labels = [], []
        y_bin = (y_class > 0).astype(np.float32)
        horizon_steps = [6, 12, 18, 24]
        lookahead     = 3
        max_offset    = max(horizon_steps) + lookahead

        # BUG-4 FIX: generate samples per segment to avoid synthetic transitions
        # at concatenation boundaries (e.g., BENIGN→attack class jumps).
        if segment_lengths is None:
            segment_lengths = [len(X)]

        start = 0
        for seg_len in segment_lengths:
            end = start + int(seg_len)
            X_seg = X[start:end]
            y_seg = y_bin[start:end]
            upper = len(X_seg) - n_timesteps - max_offset
            for i in range(0, max(0, upper)):
                window = X_seg[i:i+n_timesteps]
                fl = np.zeros(N_HORIZONS, dtype=np.float32)
                for h_idx, h in enumerate(horizon_steps):
                    fs = i + n_timesteps + h
                    fe = min(fs + lookahead, len(y_seg))
                    fl[h_idx] = float(y_seg[fs:fe].mean() > 0.5)
                self.windows.append(window)
                self.labels.append(fl)
            start = end

    def __len__(self):  return len(self.windows)
    def __getitem__(self, idx):
        return torch.FloatTensor(self.windows[idx]), torch.FloatTensor(self.labels[idx])


print(f'Model defined. Device: {DEVICE}')
# Quick param count
_m = TransformerLSTMForecaster().to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in _m.parameters()):,}')
del _m

Model defined. Device: cuda
Parameters: 341,188


In [ ]:
# ── Cell 11.5: Lightweight search space for Transformer ─────────────────────
from copy import deepcopy

RUN_LIGHT_TF_SEARCH = globals().get('RUN_LIGHT_TF_SEARCH', True)
MAX_LIGHT_TF_CONFIGS = 4

base_tf_configs = [
    dict(hidden_dim=64,  num_heads=4, num_layers=2, lstm_hidden=128, lstm_layers=2, dropout=0.2, lr=1e-3, batch_size=256, epochs=80,  patience=15),
    dict(hidden_dim=64,  num_heads=4, num_layers=3, lstm_hidden=128, lstm_layers=2, dropout=0.2, lr=5e-4, batch_size=128, epochs=100, patience=20),
    dict(hidden_dim=128, num_heads=4, num_layers=2, lstm_hidden=256, lstm_layers=2, dropout=0.3, lr=3e-4, batch_size=128, epochs=120, patience=20),
    dict(hidden_dim=96,  num_heads=4, num_layers=2, lstm_hidden=192, lstm_layers=2, dropout=0.25, lr=7e-4, batch_size=192, epochs=90, patience=18),
    dict(hidden_dim=64,  num_heads=2, num_layers=2, lstm_hidden=128, lstm_layers=1, dropout=0.2, lr=1e-3, batch_size=256, epochs=70, patience=12),
]

if RUN_LIGHT_TF_SEARCH:
    rng = np.random.default_rng(42)
    order = rng.permutation(len(base_tf_configs))[:MAX_LIGHT_TF_CONFIGS]
    TF_HP_CONFIGS_OVERRIDE = [deepcopy(base_tf_configs[i]) for i in order]
    print(f'Light TF search enabled: {len(TF_HP_CONFIGS_OVERRIDE)} candidate configs selected.')
    for i, hp in enumerate(TF_HP_CONFIGS_OVERRIDE, start=1):
        print(f'  {i}. {hp}')
else:
    print('Light TF search disabled.')

In [ ]:
# ── Cell 12: Transformer training ────────────────────────────────────────────
TARGET_TF = {'auc_h0': 0.90, 'accuracy_h0': 0.88}
VAL_RATIO = 0.1
VAL_PURGE_WINDOWS = 3

TF_HP_CONFIGS = [
    dict(hidden_dim=64,  num_heads=4, num_layers=2, lstm_hidden=128, lstm_layers=2,
         dropout=0.2, lr=1e-3, batch_size=256, epochs=80,  patience=15),
    dict(hidden_dim=64,  num_heads=4, num_layers=3, lstm_hidden=128, lstm_layers=2,
         dropout=0.2, lr=5e-4, batch_size=128, epochs=100, patience=20),
    dict(hidden_dim=128, num_heads=4, num_layers=2, lstm_hidden=256, lstm_layers=2,
         dropout=0.3, lr=3e-4, batch_size=128, epochs=120, patience=20),
]

# Workflow controls (run-once/reuse)
RUN_TF_SEARCH = True           # First run: True (search in candidate configs). Later: False.
LOAD_TF_TUNED = True           # Load saved best config if available.
SAVE_TF_TUNED = True
TF_TUNED_PATH = MODELS_DIR / 'tf_best_config.json'

if 'TF_HP_CONFIGS_OVERRIDE' in globals() and len(TF_HP_CONFIGS_OVERRIDE) > 0:
    TF_HP_CONFIGS = TF_HP_CONFIGS_OVERRIDE
    print(f'Using lightweight TF configs override: {len(TF_HP_CONFIGS)} candidate(s).')

if LOAD_TF_TUNED and TF_TUNED_PATH.exists():
    try:
        with open(TF_TUNED_PATH, 'r') as f:
            tf_loaded = json.load(f)
        if isinstance(tf_loaded, dict) and 'best_hp' in tf_loaded:
            TF_HP_CONFIGS = [tf_loaded['best_hp']]
            RUN_TF_SEARCH = False
            print(f'Loaded best TF config from {TF_TUNED_PATH}')
            print(f"  source_time={tf_loaded.get('saved_at','unknown')} | best_auc_h0={tf_loaded.get('best_auc_h0','n/a')}")
    except Exception as e:
        print(f'Failed to load saved TF config: {e}')


def split_train_val_by_segments(X, y, seg_lengths, val_ratio=0.1, purge_windows=3):
    """Temporal split inside each segment: train part + purge + validation part."""
    if seg_lengths is None:
        seg_lengths = [len(X)]

    tr_parts, val_parts = [], []
    y_tr_parts, y_val_parts = [], []
    tr_seg, val_seg = [], []

    start = 0
    for seg_len in seg_lengths:
        seg_len = int(seg_len)
        if seg_len <= 0:
            continue

        end = start + seg_len
        X_seg = X[start:end]
        y_seg = y[start:end]

        split = int(seg_len * (1 - val_ratio))
        left = max(0, split - purge_windows)
        right = min(seg_len, split + purge_windows)

        if left == 0 and seg_len > 2:
            left = max(1, seg_len // 2 - purge_windows)
        if right >= seg_len and seg_len > 2:
            right = min(seg_len - 1, max(seg_len // 2 + purge_windows, 1))

        X_tr_seg = X_seg[:left]
        y_tr_seg = y_seg[:left]
        X_val_seg = X_seg[right:]
        y_val_seg = y_seg[right:]

        if len(X_tr_seg) > 0:
            tr_parts.append(X_tr_seg)
            y_tr_parts.append(y_tr_seg)
            tr_seg.append(len(X_tr_seg))
        if len(X_val_seg) > 0:
            val_parts.append(X_val_seg)
            y_val_parts.append(y_val_seg)
            val_seg.append(len(X_val_seg))

        start = end

    if not tr_parts or not val_parts:
        raise ValueError('Unable to build non-empty train/validation split for Transformer.')

    X_tr_out = np.vstack(tr_parts).astype(np.float32)
    y_tr_out = np.hstack(y_tr_parts).astype(np.int32)
    X_val_out = np.vstack(val_parts).astype(np.float32)
    y_val_out = np.hstack(y_val_parts).astype(np.int32)
    return X_tr_out, y_tr_out, X_val_out, y_val_out, tr_seg, val_seg


def train_transformer(X_tr, y_tr, X_te, y_te, hp, seg_tr=None, seg_te=None):
    X_fit, y_fit, X_val, y_val, seg_fit, seg_val = split_train_val_by_segments(
        X_tr, y_tr, seg_tr, val_ratio=VAL_RATIO, purge_windows=VAL_PURGE_WINDOWS
    )

    train_ds = ForecastDataset(X_fit, y_fit, segment_lengths=seg_fit)
    val_ds   = ForecastDataset(X_val, y_val, segment_lengths=seg_val)
    test_ds  = ForecastDataset(X_te, y_te, segment_lengths=seg_te)
    if len(train_ds) == 0 or len(val_ds) == 0 or len(test_ds) == 0:
        print('  Empty dataset - skipping')
        return None, {}, 0, [], []

    print(f'  Train windows: {len(train_ds):,} | Val windows: {len(val_ds):,} | Test windows: {len(test_ds):,}')
    train_dl = DataLoader(train_ds, batch_size=hp['batch_size'], shuffle=True,  num_workers=2, pin_memory=True)
    val_dl   = DataLoader(val_ds,  batch_size=hp['batch_size']*2, shuffle=False, num_workers=2, pin_memory=True)
    test_dl  = DataLoader(test_ds, batch_size=hp['batch_size']*2, shuffle=False, num_workers=2, pin_memory=True)

    model = TransformerLSTMForecaster(
        hidden_dim=hp['hidden_dim'], num_heads=hp['num_heads'],
        num_layers=hp['num_layers'], lstm_hidden=hp['lstm_hidden'],
        lstm_layers=hp['lstm_layers'], dropout=hp['dropout']).to(DEVICE)

    optimizer = optim.AdamW(model.parameters(), lr=hp['lr'], weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=hp['epochs'])

    train_label_mat = np.asarray(train_ds.labels, dtype=np.float32)
    _pos_frac = train_label_mat.mean(axis=0).clip(0.01, 0.99)
    _pw = ((1 - _pos_frac) / _pos_frac).clip(0.5, 20.0)
    pos_weight = torch.tensor(_pw, dtype=torch.float32).to(DEVICE)
    print(f'  pos_weight per horizon: {_pw.round(2).tolist()}')

    use_amp = DEVICE.type == 'cuda'
    scaler_amp = GradScaler('cuda', enabled=use_amp)

    best_auc, best_state, patience_ct = 0.0, None, 0
    train_losses, val_aucs = [], []
    t0 = time.time()

    for epoch in range(hp['epochs']):
        model.train(); total_loss = 0
        for Xb, yb in train_dl:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            with autocast('cuda', enabled=use_amp):
                logits = model(Xb)
                loss = sum(
                    nn.functional.binary_cross_entropy_with_logits(
                        logits[:, h], yb[:, h], pos_weight=pos_weight[h].unsqueeze(0))
                    for h in range(N_HORIZONS)) / N_HORIZONS
            scaler_amp.scale(loss).backward()
            scaler_amp.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler_amp.step(optimizer); scaler_amp.update()
            optimizer.zero_grad(set_to_none=True)
            total_loss += loss.item()
        scheduler.step()
        train_losses.append(total_loss / len(train_dl))

        model.eval()
        val_proba = [[] for _ in range(N_HORIZONS)]
        val_labels = [[] for _ in range(N_HORIZONS)]
        with torch.no_grad():
            for Xb, yb in val_dl:
                Xb = Xb.to(DEVICE)
                with autocast('cuda', enabled=use_amp):
                    proba = torch.sigmoid(model(Xb).float())
                for h in range(N_HORIZONS):
                    val_proba[h].extend(proba[:, h].cpu().numpy())
                    val_labels[h].extend(yb[:, h].numpy())

        aucs = []
        for h in range(N_HORIZONS):
            lbl = np.array(val_labels[h]); prb = np.array(val_proba[h])
            try:
                a = roc_auc_score(lbl, prb) if len(np.unique(lbl)) > 1 else 0.5
            except Exception:
                a = 0.5
            aucs.append(a)
        mean_auc = float(np.mean(aucs))
        val_aucs.append(mean_auc)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            auc_str = '  '.join([f'{HORIZON_LABELS[h]}:{aucs[h]:.3f}' for h in range(N_HORIZONS)])
            print(f'  Ep {epoch+1:3d}/{hp["epochs"]} | loss={train_losses[-1]:.4f} | VAL AUC [{auc_str}]')

        if mean_auc > best_auc:
            best_auc = mean_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_ct = 0
        else:
            patience_ct += 1
            if patience_ct >= hp['patience']:
                print(f'  Early stop at epoch {epoch+1}')
                break

    if best_state:
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

    model.eval()
    all_proba = [[] for _ in range(N_HORIZONS)]
    all_labels_h = [[] for _ in range(N_HORIZONS)]
    with torch.no_grad():
        for Xb, yb in test_dl:
            Xb = Xb.to(DEVICE)
            with autocast('cuda', enabled=use_amp):
                proba = torch.sigmoid(model(Xb).float())
            for h in range(N_HORIZONS):
                all_proba[h].extend(proba[:, h].cpu().numpy())
                all_labels_h[h].extend(yb[:, h].numpy())

    final_aucs, final_accs = [], []
    for h in range(N_HORIZONS):
        lbl = np.array(all_labels_h[h]); prb = np.array(all_proba[h])
        try:
            a = roc_auc_score(lbl, prb) if len(np.unique(lbl)) > 1 else 0.5
        except Exception:
            a = 0.5
        final_aucs.append(a)
        final_accs.append(float(((prb > 0.5).astype(int) == lbl.astype(int)).mean()))

    metrics = {
        'auc_h0': final_aucs[0], 'accuracy_h0': final_accs[0],
        'auc_all': final_aucs, 'acc_all': final_accs, 'auc_mean': float(np.mean(final_aucs))
    }
    return model, metrics, time.time()-t0, train_losses, val_aucs


best_tf, best_tf_metrics, best_tf_losses, best_tf_aucs = None, {}, [], []
best_tf_hp = None

candidates = TF_HP_CONFIGS if RUN_TF_SEARCH else TF_HP_CONFIGS[:1]
for attempt, hp in enumerate(candidates):
    print(f'\n--- Transformer Attempt {attempt+1}/{len(candidates)}: {hp} ---')
    model, metrics, elapsed, losses, aucs = train_transformer(
        X_tf_train, y_tf_train, X_tf_test, y_tf_test, hp,
        seg_tr=tf_train_segment_lengths, seg_te=tf_test_segment_lengths
    )
    if model is None:
        continue
    print(f'  auc_h0={metrics["auc_h0"]:.4f}  acc_h0={metrics["accuracy_h0"]:.4f}  '
          f'auc_mean={metrics["auc_mean"]:.4f}  time={elapsed:.1f}s')

    if not best_tf_metrics or metrics['auc_h0'] > best_tf_metrics.get('auc_h0', 0):
        best_tf, best_tf_metrics, best_tf_losses, best_tf_aucs = model, metrics, losses, aucs
        best_tf_hp = hp

    if RUN_TF_SEARCH and all(metrics.get(k, 0) >= v for k, v in TARGET_TF.items()):
        print('All targets met!')
        break

if SAVE_TF_TUNED and best_tf_hp is not None:
    payload = {
        'saved_at': time.strftime('%Y-%m-%d %H:%M:%S'),
        'best_auc_h0': float(best_tf_metrics.get('auc_h0', 0.0)),
        'best_acc_h0': float(best_tf_metrics.get('accuracy_h0', 0.0)),
        'best_hp': best_tf_hp,
    }
    with open(TF_TUNED_PATH, 'w') as f:
        json.dump(payload, f, indent=2)
    print(f'Saved best TF config to {TF_TUNED_PATH}')

print(f'\nBEST Transformer: auc_h0={best_tf_metrics["auc_h0"]:.4f}  '
      f'acc_h0={best_tf_metrics["accuracy_h0"]:.4f}  auc_mean={best_tf_metrics["auc_mean"]:.4f}')


--- Transformer Attempt 1/3: {'hidden_dim': 64, 'num_heads': 4, 'num_layers': 2, 'lstm_hidden': 128, 'lstm_layers': 2, 'dropout': 0.2, 'lr': 0.001, 'batch_size': 256, 'epochs': 80, 'patience': 15} ---
  Train windows: 176,334 | Test windows: 44,015
  pos_weight per horizon: [0.5, 0.5, 0.5, 0.5]
  Ep   1/80 | loss=0.0178 | AUC [+6w:0.995  +12w:0.991  +18w:0.989  +24w:0.985]
  Ep  10/80 | loss=0.0059 | AUC [+6w:0.992  +12w:0.990  +18w:0.989  +24w:0.985]
  Ep  20/80 | loss=0.0053 | AUC [+6w:0.994  +12w:0.992  +18w:0.990  +24w:0.987]
  Ep  30/80 | loss=0.0052 | AUC [+6w:0.995  +12w:0.992  +18w:0.990  +24w:0.987]
  Early stop at epoch 31
  auc_h0=0.9948  acc_h0=0.9977  auc_mean=0.9911  time=351.1s
All targets met!

BEST Transformer: auc_h0=0.9948  acc_h0=0.9977  auc_mean=0.9911


In [15]:
# ── Cell 13: Transformer training curves + save ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(best_tf_losses, color='steelblue')
axes[0].set_title('Train Loss (BCE per horizon)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')

axes[1].plot(best_tf_aucs, color='coral')
axes[1].axhline(TARGET_TF['auc_h0'], color='green', linestyle='--', label=f'Target {TARGET_TF["auc_h0"]}')
axes[1].set_title('Val AUC (mean across 4 horizons)')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('AUC')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / '05_transformer_training.png', dpi=150)
plt.show()

# Per-horizon AUC bar chart
fig, ax = plt.subplots(figsize=(7, 4))
auc_vals = best_tf_metrics['auc_all']
bars = ax.bar(HORIZON_LABELS, auc_vals, color=['#2196F3','#4CAF50','#FF9800','#E91E63'])
ax.axhline(TARGET_TF['auc_h0'], color='red', linestyle='--', label=f'Target {TARGET_TF["auc_h0"]}')
ax.set_ylim(0.5, 1.0); ax.set_title('AUC per Forecast Horizon')
ax.set_ylabel('AUC (ROC)'); ax.legend()
for bar, v in zip(bars, auc_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f'{v:.4f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '06_horizon_auc.png', dpi=150)
plt.show()

# Save Transformer model
tf_path = MODELS_DIR / 'transformer_v3.pt'
torch.save(best_tf.state_dict(), str(tf_path))
with open(MODELS_DIR / 'transformer_metrics.json', 'w') as f:
    json.dump(best_tf_metrics, f, indent=2)
print(f'Saved: {tf_path}')
with open(MODELS_DIR / 'scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

Saved: /kaggle/working/pad_onap_v3/models/transformer_v3.pt


---
## Section 5 — Final Summary

In [17]:
# ── Cell 14: Final summary ───────────────────────────────────────────────────
print('=' * 65)
print('PAD-ONAP M2 - TRAINING COMPLETE')
print('=' * 65)
print()
print(f'DATASET (full scan):')
print(f'  Total rows read : {total_rows_read:>12,}')
print(f'  BENIGN rows     : {total_benign_rows:>12,}')
print(f'  Attack rows     : {grand_attack:>12,}')
print(f'  Raw BENIGN wins : {n_raw:>12,}')
print(f'  Total windows   : {len(X_train) + len(X_test):>12,}  (train+test)')
print()
print('VALIDATION SETTINGS:')
print(f'  Temporal split ratio : {TEMPORAL_SPLIT_RATIO}')
print(f'  Purge windows        : {PURGE_WINDOWS}')
print(f'  Max BENIGN oversample: x{MAX_BENIGN_OVERSAMPLE} (train only)')
print('  Test BENIGN          : no oversampling')
print()
print(f'TRACK A - XGBoost 7-class:')
print(f'  Accuracy     : {best_xgb_metrics["accuracy"]:.4f}  (target >= {TARGET_XGB["accuracy"]})')
print(f'  Balanced Acc : {best_xgb_metrics["balanced_acc"]:.4f}  (target >= {TARGET_XGB["balanced_acc"]})')
print(f'  Macro F1     : {best_xgb_metrics["macro_f1"]:.4f}  (target >= {TARGET_XGB["macro_f1"]})')
print(f'  AUC (OvR)    : {best_xgb_metrics["auc_macro_ovr"]:.4f}')
print(f'  Saved        : {MODELS_DIR}/xgboost_v3.json')
print()
print(f'TRACK B - Transformer+LSTM 4-horizon:')
for h, (auc, acc) in enumerate(zip(best_tf_metrics['auc_all'], best_tf_metrics['acc_all'])):
    print(f'  {HORIZON_LABELS[h]}: AUC={auc:.4f}  Acc={acc:.4f}')
print(f'  Saved        : {MODELS_DIR}/transformer_v3.pt')
print()
print(f'OUTPUTS saved to: {OUTPUT_DIR}')
print()

# Check targets
a_ok = all(best_xgb_metrics.get(k, 0) >= v for k, v in TARGET_XGB.items())
b_ok = all(best_tf_metrics.get(k, 0) >= v for k, v in TARGET_TF.items())
print(f'Track A targets met: {"YES" if a_ok else "NO"}')
print(f'Track B targets met: {"YES" if b_ok else "NO"}')

PAD-ONAP M2 - TRAINING COMPLETE

DATASET (full scan):
  Total rows read :   19,125,130
  BENIGN rows     :      113,828
  Attack rows     :   70,313,809
  Raw BENIGN wins :        1,359
  Total windows   :      230,343  (train+test)

VALIDATION SETTINGS:
  Temporal split ratio : 0.8
  Purge windows        : 10
  Max BENIGN oversample: x20 (train only)
  Test BENIGN          : no oversampling

TRACK A - XGBoost 7-class:
  Accuracy     : 0.9090  (target >= 0.95)
  Balanced Acc : 0.9516  (target >= 0.9)
  Macro F1     : 0.9384  (target >= 0.9)
  AUC (OvR)    : 0.9940
  Saved        : /kaggle/working/pad_onap_v3/models/xgboost_v3.json

TRACK B - Transformer+LSTM 4-horizon:
  +6w: AUC=0.9948  Acc=0.9977
  +12w: AUC=0.9921  Acc=0.9975
  +18w: AUC=0.9901  Acc=0.9970
  +24w: AUC=0.9873  Acc=0.9967
  Saved        : /kaggle/working/pad_onap_v3/models/transformer_v3.pt

OUTPUTS saved to: /kaggle/working/pad_onap_v3

Track A targets met: NO
Track B targets met: YES


In [18]:
# ── Cell 15: Package outputs for download ─────────────────────────────────────
import shutil
shutil.make_archive('/kaggle/working/pad_onap_v3_models', 'zip', str(MODELS_DIR))
print('Created: /kaggle/working/pad_onap_v3_models.zip')

# List all saved files
print('\nAll output files:')
for p in sorted(OUTPUT_DIR.rglob('*')):
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        print(f'  {str(p.relative_to(OUTPUT_DIR)):<45} {size_kb:>8.1f} KB')

Created: /kaggle/working/pad_onap_v3_models.zip

All output files:
  figures/01_class_distribution_raw.png             75.5 KB
  figures/02_train_test_dist.png                    67.3 KB
  figures/03_xgb_confusion_matrix.png               67.4 KB
  figures/04_shap_summary.png                      100.1 KB
  figures/05_transformer_training.png               60.8 KB
  figures/06_horizon_auc.png                        32.6 KB
  models/scaler.pkl                                  0.8 KB
  models/transformer_metrics.json                    0.3 KB
  models/transformer_v3.pt                        1371.3 KB
  models/xgb_label_map.json                          0.3 KB
  models/xgboost_v3.json                          3529.1 KB
  processed/X_benign_raw.npy                        90.4 KB
  processed/X_test.npy                            2935.9 KB
  processed/X_train.npy                          12360.5 KB
  processed/metadata.json                            1.4 KB
  processed/scaler.pkl           